<a href="https://colab.research.google.com/github/Sanjivkumar100/Emotion_Aware_Production_System/blob/main/Product_and_Emotion_Full_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import gradio as gr
product_df=pd.read_csv('/content/drive/MyDrive/Emotional - Product Dataset/processed_flipkart_products.csv')

In [ ]:
import pickle as pkl
import tensorflow as tf
from tensorflow.keras.models import load_model

with open('/content/drive/MyDrive/Emotional - Product Dataset/tokenizer.pkl', 'rb') as f:
  tokenizer=pkl.load(f)
with open('/content/drive/MyDrive/Emotional - Product Dataset/label_encoder.pkl', 'rb') as f:
  label_encoder=pkl.load(f)

classification=load_model('/content/drive/MyDrive/Emotional - Product Dataset/emotion_model_classification.h5')

# Load the regression model with compile=False and then manually compile it
regression_model=load_model('/content/drive/MyDrive/Emotional - Product Dataset/emotion_reg.h5', compile=False)
regression_model.compile(optimizer='adam', loss='mse', metrics=['mse'])

In [ ]:
product_df.columns

Index(['product_name', 'product_category_tree', 'description', 'brand',
       'retail_price', 'category', 'clean_description', 'normalized_category'],
      dtype='object')

In [ ]:
product_df['normalized_category'].unique()

array(['fashion', 'home', 'other', 'stationery', 'automotive', 'fitness',
       'kids', 'electronics', 'beauty'], dtype=object)

In [ ]:
product_df["normalized_category"] = product_df["normalized_category"].replace(
    {"stationery": "books"}
)


In [ ]:
MAX_LEN = 120
import numpy as np
def predict_intensity(text):

    seq = tokenizer.texts_to_sequences([text])
    padded = tf.keras.preprocessing.sequence.pad_sequences(seq, maxlen=MAX_LEN)

    intensity = regression_model.predict(padded)[0][0]

    return float(intensity)
def predict_emotion(text):

    seq = tokenizer.texts_to_sequences([text])
    padded = tf.keras.preprocessing.sequence.pad_sequences(seq, maxlen=MAX_LEN)

    pred = classification.predict(padded)

    emotion = label_encoder.inverse_transform([np.argmax(pred)])

    return emotion[0]



In [ ]:
emotion_category_map = {

    "joy": ["fashion","electronics","fitness"],

    "sadness": ["home","beauty","books"],

    "anger": ["fitness","electronics"],

    "fear": ["home","beauty"],

    "neutral": ["electronics","home","fashion"]
}


def decide_top_n(intensity):
  if intensity >=0.75:
    return 10
  elif intensity >=0.5:
    return 5
  else:
    return 3
def recommend_products(emotion, intensity):

    categories = emotion_category_map.get(emotion.lower(), ["other"])

    top_n = decide_top_n(intensity)

    filtered = product_df[
        product_df["normalized_category"].isin(categories)
    ]

    recommendations = filtered.sample(min(top_n, len(filtered)))

    return recommendations[["product_name","normalized_category"]] \
            .rename(columns={"normalized_category":"category"})


In [ ]:
def emotion_product_pipeline(text):

    emotion = predict_emotion(text)
    intensity = predict_intensity(text)

    products = recommend_products(emotion, intensity)

    return emotion, intensity, products


In [ ]:
emotion, intensity, recs = emotion_product_pipeline(
    "I feel very unhappy today"
)

print("Emotion:", emotion)
print("Intensity:", intensity)
print(recs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step
Emotion: sadness
Intensity: 0.5116023421287537
                                            product_name category
687    SHEFFIELD CLASSIC SH-83-VC1 Hand-held Vacuum C...     home
3922                                    Ndura Kadhai 1 L     home
4276                         NAVISHA Plastic Hand Juicer     home
13465                Shubham Exports Showpiece  -  28 cm     home
10794                     Sea Shell Floral pattern CS102     home


In [ ]:
interface = gr.Interface(
    fn=emotion_product_pipeline,
    inputs=gr.Textbox(lines=3, placeholder="Enter your text here..."),
    outputs=[
        gr.Text(label="Predicted Emotion"),
        gr.Number(label="Emotion Intensity"),
        gr.Dataframe(label="Recommended Products")
    ],
    title="Emotion-Aware Product Recommendation System",
    description="Enter your text. The system predicts emotion, intensity, and suggests relevant products."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a0344804f54fab37a0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
